# Model Inference (Local Pipeline & Live API Check)

This notebook demonstrates the assignment's model-packaging and inference deliverable: it loads the saved preprocessing-and-classifier pipeline, predicts a binary class from the same 13 patient-feature fields accepted by the FastAPI service, and optionally sends the same example to the running `/predict` endpoint.

## Execution Prerequisites (Read Before Running)

* **Cell 1 (Local Artifact Inference — No Server Required):** Loads `models/heart_disease_pipeline.joblib` directly from disk. If the model file is missing, run `python train.py` or `02_model_training_and_tracking.ipynb` first.
* **Cell 2 (Live FastAPI Endpoint Check — Server Required):** Sends an HTTP `POST` request to `http://127.0.0.1:8000/predict`. **Do not click "Run All" unless the FastAPI server is already running in a separate terminal**, otherwise Cell 2 will raise a connection error.

> **Disclaimer:** The included record is an example payload for demonstrating the code, not a real patient's record. Model predictions and probabilities are educational outputs only; they are not medical advice, a diagnosis, or a clinically validated risk estimate.

In [3]:
from pathlib import Path

import joblib
import pandas as pd

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "train.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate the project root containing train.py.")
model_path = project_root / "models" / "heart_disease_pipeline.joblib"
if not model_path.is_file():
    raise FileNotFoundError("Model artifact is missing. Run the training notebook or python train.py.")

model = joblib.load(model_path)
feature_columns = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal",
]
patient = {
    "age": 52,
    "sex": 1,
    "cp": 2,
    "trestbps": 125,
    "chol": 212,
    "fbs": 0,
    "restecg": 1,
    "thalach": 168,
    "exang": 0,
    "oldpeak": 1.0,
    "slope": 2,
    "ca": 2,
    "thal": 3,
}
if set(patient) != set(feature_columns):
    raise ValueError("Patient input fields do not match the model's 13 expected features.")
patient_frame = pd.DataFrame([patient], columns=feature_columns)
prediction = int(model.predict(patient_frame)[0])
positive_probability = float(model.predict_proba(patient_frame)[0, 1])
result = {
    "prediction": prediction,
    "probability": positive_probability,
    "label": "Heart disease predicted" if prediction == 1 else "No heart disease predicted",
}
display(pd.Series(result, name="prediction result"))
print("Educational model output only; this is not medical advice or a diagnosis.")

prediction                              0
probability                      0.495482
label          No heart disease predicted
Name: prediction result, dtype: object

Educational model output only; this is not medical advice or a diagnosis.


## Step 2 (Optional): Request a Prediction from the Running FastAPI Server

> **IMPORTANT — Start the API Server First:**
> Before running the code cell below, open a separate PowerShell terminal at the `heart-disease-mlops` project root, activate `.venv`, and start the FastAPI service:
>
> ```powershell
> .\.venv\Scripts\Activate.ps1
> python -m uvicorn api.main:app --reload --no-access-log --host 127.0.0.1 --port 8000
> ```
>
> Leave that terminal open while executing the cell below. If the API server is not running on port `8000`, the cell will intentionally stop and report a connection error instead of silently substituting a local prediction.

The API validates all 13 feature ranges/category codes and returns a binary `prediction` (`0` or `1`), the positive-class `probability` (`0.0` to `1.0`), and a human-readable `label`. The probability is the model's estimated probability for class 1 under this dataset and pipeline; it is not a calibrated clinical confidence or diagnosis.

In [4]:
import json
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

# Step 1: Send the sample patient payload as JSON to the live FastAPI /predict endpoint.
api_request = Request(
    "http://127.0.0.1:8000/predict",
    data=json.dumps(patient).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)
try:
    with urlopen(api_request, timeout=10) as response:
        api_result = json.loads(response.read().decode("utf-8"))
except HTTPError as error:
    detail = error.read().decode("utf-8", errors="replace")
    raise RuntimeError(f"API returned HTTP {error.code}: {detail}") from error
except URLError as error:
    raise RuntimeError(
        "Could not connect to the API. Start it with: "
        "python -m uvicorn api.main:app --reload --no-access-log --host 127.0.0.1 --port 8000"
    ) from error

# Step 2: Validate the response schema and probability bounds.
required_fields = {"prediction", "probability"}
if not required_fields.issubset(api_result):
    raise ValueError(f"API response is missing required fields: {required_fields - api_result.keys()}")
if api_result["prediction"] not in (0, 1) or not 0 <= api_result["probability"] <= 1:
    raise ValueError("API returned an invalid prediction or probability.")
display(pd.Series(api_result, name="API response"))

prediction                              0
probability                      0.495482
label          No heart disease predicted
Name: API response, dtype: object